In [1]:
import librosa
import numpy as np
from pydub import AudioSegment
import os
import torch
import librosa
from pyannote.audio import Model
from pyannote.audio.pipelines import VoiceActivityDetection

c:\Users\Siddharth\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
c:\Users\Siddharth\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyannote\audio\core\io.py:48: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your 

In [ ]:
audio = AudioSegment.from_file(r"D:\VADBASIC\260217_0025_1-2.wav")

In [ ]:
HF_TOKEN = "." #your hf token here

speaker_files = {
    "spk1": r"D:\VADBASIC\260217_0025_3.wav",
    "spk2": r"D:\VADBASIC\260217_0025_4.wav",
}

session_id = "260217_0025_1-2"
output_rttm = rf"D:\VADBASIC\{session_id}.rttm"

In [4]:
model = Model.from_pretrained("pyannote/segmentation-3.0", token=HF_TOKEN)

vad_pipeline = VoiceActivityDetection(segmentation=model)
vad_pipeline.instantiate({
    "min_duration_on": 0.25,
    "min_duration_off": 0.1
})

In [5]:
def get_segments(path):
    wav, sr = librosa.load(path, sr=16000, mono=True)
    waveform = torch.from_numpy(wav).unsqueeze(0) 
    audio_dict = {"waveform": waveform, "sample_rate": sr}
    output = vad_pipeline(audio_dict)
    return [(seg.start, seg.end) for seg in output.get_timeline()]

In [6]:
lines = []
for spk, path in speaker_files.items():
    segs = get_segments(path)
    print(f"{spk}: {len(segs)} segments")
    for start, end in segs:
        dur = end - start
        lines.append((start, f"SPEAKER {session_id} 1 {start:.3f} {dur:.3f} <NA> <NA> {spk} <NA> <NA>"))

lines.sort(key=lambda x: x[0])

with open(output_rttm, "w") as f:
    f.write("\n".join(l for _, l in lines) + "\n")

print(f"\n{len(lines)} total segments written to {output_rttm}")

spk1: 946 segments
spk2: 925 segments

1871 total segments written to D:\VADBASIC\260217_0025_1-2.rttm


In [ ]:
import huggingface_hub, pyannote.audio
print(huggingface_hub.__version__)
print(pyannote.audio.__version__)

0.25.2
4.0.7
